# T Cell Analysis: CD4/CD8 Classification + scANVI Re-training

## Pipeline Overview

**Key Modifications from Original:**
1. ✅ **Subset out Mono-mac cells** (using `cell_type_celltypist_filt` column)
2. ✅ **CD4/CD8 T cell classification** (based on CD4, CD8A expression)
3. ✅ **Combined labels** (e.g., "CD4+ Tem", "CD8+ Effector helper T cells")
4. ✅ **Re-train scANVI** (skip scVI, use existing X_scvi)

**Input:** T cell data with existing scVI and CellTypist annotations

**Output:** adata_tcell_cd4cd8_FINAL.h5ad with CD4/CD8-aware annotations

---

Author: r2end  
Date: 2025-01-09  
Version: 1.0

## Imports and Configuration

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Core libraries
import numpy as np
import pandas as pd
from pathlib import Path
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import time
import gc

# scvi-tools
import scvi
import torch

print("Libraries imported successfully")

## Configuration Parameters

In [ ]:
# ============================================================================
# INPUT/OUTPUT PATHS
# ============================================================================

# Input: Existing T cell data with scVI + CellTypist annotations
INPUT_FILE = "/home/h2048/data/py/0108/tcell_scanvi_v3_5_1_hotfix/adata_epithelial_FINAL.h5ad"

# Output directory
OUTPUT_DIR = "/home/h2048/data/py/0109/tcell_cd4cd8_scanvi"

# ============================================================================
# MONOCYTE-MACROPHAGE FILTERING
# ============================================================================

# Cell types to remove (keywords to match in cell_type_celltypist_filt)
MONOMAC_KEYWORDS = ['Monocyte', 'Mono', 'Macrophage', 'Mac', 'myeloid', 'Mono-mac']

# ============================================================================
# CD4/CD8 CLASSIFICATION
# ============================================================================

CD4_GENE = 'CD4'
CD8_GENE = 'CD8A'
CD4CD8_THRESHOLD = 1.0  # log1p normalized expression threshold

# Classification rules:
# - CD4 > threshold & CD8 < threshold → CD4+
# - CD8 > threshold & CD4 < threshold → CD8+
# - Both high → Double Positive (DP) - rare in periphery
# - Both low → Double Negative (DN) - includes NK cells

# ============================================================================
# SCANVI TRAINING (RE-TRAINING ONLY)
# ============================================================================

# Load existing scVI model
SCVI_MODEL_PATH = "/home/h2048/data/py/0108/tcell_scanvi_v3_5_1_hotfix/models/scvi_model"

SCANVI_MAX_EPOCHS = 600
BATCH_SIZE = 2048
LEARNING_RATE = 1e-3
EARLY_STOPPING = True
EARLY_STOPPING_PATIENCE = 50

# ============================================================================
# RARE TYPE FILTERING
# ============================================================================

MIN_CELLS_PER_TYPE = 200  # Types with <200 cells → Unknown

# ============================================================================
# COLUMN KEYS
# ============================================================================

BATCH_KEY = 'dataset'
CD4CD8_KEY = 'cd4_cd8_type'
CELLTYPIST_KEY = 'cell_type_celltypist_filt'  # Use filtered CellTypist labels
COMBINED_LABEL_KEY = 'cell_type_combined'  # CD4/CD8 + CellTypist
SCANVI_LABEL_KEY = 'cell_type_scanvi_cd4cd8'

# ============================================================================
# RANDOM SEED
# ============================================================================

RANDOM_SEED = 42

print("="*80)
print("T CELL CD4/CD8 + scANVI RE-TRAINING PIPELINE")
print("="*80)
print(f"\nConfiguration:")
print(f"  Input: {INPUT_FILE}")
print(f"  Output: {OUTPUT_DIR}")
print(f"  scVI model: {SCVI_MODEL_PATH}")
print(f"  Batch key: {BATCH_KEY}")
print(f"  CD4/CD8 threshold: {CD4CD8_THRESHOLD}")
print(f"  Rare type threshold: {MIN_CELLS_PER_TYPE} cells")

## Helper Functions

In [ ]:
def merge_rare_types(s: pd.Series, min_cells: int = 200, other: str = "Unknown") -> pd.Series:
    """
    Merge rare cell types (< min_cells) into 'other' category.
    """
    s = s.astype(str).copy()
    vc = s.value_counts()
    rare = vc[vc < min_cells].index
    
    if len(rare) > 0:
        print(f"   Merging {len(rare)} rare types (<{min_cells} cells) → {other}")
        for rt in rare[:5]:
            print(f"      {rt}: {vc[rt]} cells")
        if len(rare) > 5:
            print(f"      ... and {len(rare)-5} more")
        s.loc[s.isin(rare)] = other
    
    return s


def save_celltype_counts(counts_dict: dict, output_dir: Path) -> None:
    """Save cell type count statistics to CSV files"""
    for name, series in counts_dict.items():
        df = pd.DataFrame({
            'cell_type': series.index,
            'count': series.values,
            'percentage': (series.values / series.sum() * 100).round(2)
        })
        df = df.sort_values('count', ascending=False)
        filepath = output_dir / f"{name}.csv"
        df.to_csv(filepath, index=False)
        print(f"   ✓ Saved: {filepath.name}")


print("Helper functions defined")

## Set Random Seeds

In [ ]:
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

scvi.settings.seed = RANDOM_SEED

print(f"✓ Random seed set to {RANDOM_SEED}")

## Environment Setup

In [ ]:
# GPU detection
print(f"\n{'='*80}")
print("GPU CHECK")
print("="*80)

GPU_AVAILABLE = torch.cuda.is_available()
print(f"GPU available: {GPU_AVAILABLE}")

if GPU_AVAILABLE:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name}")
    print(f"Memory: {gpu_memory:.1f} GB")
    accelerator = 'gpu'
    devices = 1
else:
    print("⚠️  Running on CPU (training will be slow)")
    accelerator = 'cpu'
    devices = 'auto'

# Create output directories
OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FIG_DIR = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)

MODEL_DIR = OUTPUT_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)

print(f"\n✓ Output directories created")

# Configure scanpy
sc.settings.verbosity = 3
sc.settings.n_jobs = 48
sc.settings.figdir = FIG_DIR
sc.set_figure_params(dpi=300, facecolor='white', format='pdf')

## STAGE 0: Data Loading

In [ ]:
print(f"\n{'='*80}")
print("STAGE 0: DATA LOADING")
print("="*80)

print(f"\nLoading: {INPUT_FILE}")
adata = sc.read_h5ad(INPUT_FILE)

print(f"✓ Data loaded:")
print(f"  Cells: {adata.n_obs:,}")
print(f"  Genes: {adata.n_vars:,}")

# Check required columns
required_cols = [BATCH_KEY, CELLTYPIST_KEY]
missing = [col for col in required_cols if col not in adata.obs.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Check for X_scvi
if 'X_scvi' not in adata.obsm:
    raise ValueError("X_scvi not found in adata.obsm. Need pre-trained scVI model.")

print(f"\n✓ Required data structures found:")
print(f"  Batch key: {BATCH_KEY}")
print(f"  CellTypist labels: {CELLTYPIST_KEY}")
print(f"  X_scvi: {adata.obsm['X_scvi'].shape}")
print(f"  Layers: {list(adata.layers.keys())}")
print(f"  .raw: {adata.raw is not None}")

In [ ]:
# ============================================================================
# INSERT THIS CELL RIGHT AFTER STAGE 0 (after log1p creation)
# BEFORE STAGE 1 (cell filtering)
# ============================================================================
# Force include T cell core markers in HVG for downstream analysis

print(f"\n{'='*80}")
print("FORCE INCLUDE T CELL MARKERS IN HVG")
print("="*80)

# Define T cell core markers
TCELL_CORE_MARKERS = {
    'Pan_T': ['CD3D', 'CD3E', 'CD3G'],
    'CD4_T': ['CD4', 'CD40LG'],
    'CD8_T': ['CD8A', 'CD8B'],
    'NK': ['GNLY', 'NKG7', 'KLRD1', 'FCGR3A'],
    'Naive': ['CCR7', 'TCF7', 'LEF1', 'SELL', 'IL7R'],
    'Memory': ['GZMK', 'CD69'],
    'Effector': ['GZMB', 'GZMH', 'PRF1', 'IFNG'],
    'Treg': ['FOXP3', 'IL2RA', 'IKZF2', 'TNFRSF4'],
    'Exhausted': ['PDCD1', 'HAVCR2', 'LAG3', 'TIGIT'],
    'Proliferating': ['MKI67', 'TOP2A', 'STMN1'],
    'Th2': ['GATA3', 'IL4', 'IL5', 'IL13'],
    'Th17': ['RORC', 'IL17A', 'IL23R'],
}

# Flatten all markers
all_markers = []
for category, genes in TCELL_CORE_MARKERS.items():
    all_markers.extend(genes)
all_markers = list(set(all_markers))  # Remove duplicates

print(f"\nTotal unique T cell markers to check: {len(all_markers)}")

# Check if highly_variable column exists
if 'highly_variable' not in adata.var.columns:
    print(f"\n⚠️  'highly_variable' column not found, creating it...")
    # Compute HVG if not exists
    sc.pp.highly_variable_genes(
        adata, 
        n_top_genes=4000,
        layer='counts',
        batch_key=BATCH_KEY if BATCH_KEY in adata.obs.columns else None,
        subset=False
    )
    print(f"   ✓ Created 'highly_variable' column")

# Get gene symbol column
gene_symbols = None
symbol_col = None
for col in ['symbol_base', 'gene_symbol', 'gene_symbols', 'feature_name']:
    if col in adata.var.columns:
        gene_symbols = adata.var[col].astype(str)
        symbol_col = col
        break

if gene_symbols is None:
    gene_symbols = adata.var_names.astype(str)
    symbol_col = 'var_names'

print(f"\nUsing gene symbols from: {symbol_col}")
print(f"Total genes in adata: {adata.n_vars}")
print(f"Current HVG count: {adata.var['highly_variable'].sum()}")

# Check which markers are present and their HVG status
print(f"\n{'='*60}")
print("Checking T cell markers in data:")
print("="*60)

missing_in_data = []
missing_in_hvg = []
present_in_hvg = []

for category, genes in TCELL_CORE_MARKERS.items():
    print(f"\n{category}:")
    for gene in genes:
        # Find gene (case-insensitive)
        matches = gene_symbols.str.upper() == gene.upper()
        
        if matches.sum() == 0:
            print(f"  ✗ {gene}: NOT IN DATA")
            missing_in_data.append(gene)
        else:
            # Get first match index
            idx = matches[matches].index[0]
            is_hvg = adata.var.loc[idx, 'highly_variable']
            
            if is_hvg:
                print(f"  ✓ {gene}: in HVG")
                present_in_hvg.append(gene)
            else:
                print(f"  ⚠️  {gene}: NOT in HVG (will add)")
                missing_in_hvg.append(gene)

# Summary
print(f"\n{'='*60}")
print(f"SUMMARY:")
print(f"  Total markers checked: {len(all_markers)}")
print(f"  Present in data: {len(present_in_hvg) + len(missing_in_hvg)}")
print(f"  Missing in data: {len(missing_in_data)}")
print(f"  Already in HVG: {len(present_in_hvg)}")
print(f"  NEED TO ADD to HVG: {len(missing_in_hvg)}")
print("="*60)

if len(missing_in_data) > 0:
    print(f"\n⚠️  Markers missing in data: {missing_in_data}")
    print(f"   These markers cannot be used for classification")

# Force add missing markers to HVG
if len(missing_in_hvg) > 0:
    print(f"\n🔧 Force adding {len(missing_in_hvg)} markers to HVG...")
    
    for gene in missing_in_hvg:
        matches = gene_symbols.str.upper() == gene.upper()
        if matches.sum() > 0:
            idx = matches[matches].index[0]
            adata.var.loc[idx, 'highly_variable'] = True
            print(f"   ✓ Added {gene} to HVG")
    
    new_hvg_count = adata.var['highly_variable'].sum()
    print(f"\n✓ Updated HVG count: {new_hvg_count} (added {new_hvg_count - 4000})")
else:
    print(f"\n✓ All required markers already in HVG")

# Verify critical markers for CD4/CD8 classification
critical_markers = ['CD4', 'CD8A', 'CD8B']
print(f"\n{'='*60}")
print(f"CRITICAL: Verify CD4/CD8 markers:")
print("="*60)

for gene in critical_markers:
    matches = gene_symbols.str.upper() == gene.upper()
    if matches.sum() > 0:
        idx = matches[matches].index[0]
        is_hvg = adata.var.loc[idx, 'highly_variable']
        status = "✓ IN HVG" if is_hvg else "✗ NOT IN HVG"
        print(f"  {gene}: {status}")
    else:
        print(f"  {gene}: ✗ NOT IN DATA")

print("="*60)
print(f"\n✅ T cell marker check complete")
print(f"   HVG updated: {adata.var['highly_variable'].sum()} genes")
print("="*60)

## STAGE 1: Filter Out Monocyte-Macrophage Cells

In [ ]:
print(f"\n{'='*80}")
print("STAGE 1: FILTER MONOCYTE-MACROPHAGE CELLS")
print("="*80)

# Get cell type counts before filtering
print(f"\nCell type distribution (before filtering):")
celltypist_counts = adata.obs[CELLTYPIST_KEY].value_counts()
print(celltypist_counts.head(20))

# Identify Mono-mac cells by keyword matching
print(f"\nIdentifying Mono-mac cells (keywords: {MONOMAC_KEYWORDS})...")

monomac_mask = adata.obs[CELLTYPIST_KEY].astype(str).str.lower().str.contains(
    '|'.join([kw.lower() for kw in MONOMAC_KEYWORDS]),
    case=False,
    na=False
)

n_monomac = monomac_mask.sum()
pct_monomac = n_monomac / adata.n_obs * 100

print(f"\nMono-mac cells identified: {n_monomac:,} ({pct_monomac:.1f}%)")

if n_monomac > 0:
    print(f"\nMono-mac cell types to remove:")
    monomac_types = adata.obs.loc[monomac_mask, CELLTYPIST_KEY].value_counts()
    for ct, count in monomac_types.items():
        print(f"  {ct}: {count:,} cells")

# Filter out Mono-mac cells
print(f"\nFiltering out Mono-mac cells...")
adata = adata[~monomac_mask].copy()

print(f"✓ Data after Mono-mac filtering:")
print(f"  Cells: {adata.n_obs:,}")
print(f"  Genes: {adata.n_vars:,}")
print(f"  Removed: {n_monomac:,} cells ({pct_monomac:.1f}%)")

# Show remaining cell types
print(f"\nRemaining cell type distribution:")
remaining_counts = adata.obs[CELLTYPIST_KEY].value_counts()
print(remaining_counts.head(20))

## STAGE 2: CD4/CD8 T Cell Classification

In [ ]:
# ============================================================================
# STAGE 2: CD4/CD8 CLASSIFICATION (USE FULL ADATA, NOT HVG SUBSET)
# ============================================================================
# REPLACE the original STAGE 2 with this version

print(f"\n{'='*80}")
print("STAGE 2: CD4/CD8 CLASSIFICATION (FULL GENE SET)")
print("="*80)

# Get gene symbol column (from FULL adata, not HVG subset)
print(f"\nLocating gene symbols in FULL adata...")

gene_symbols = None
symbol_col = None

for col in ['symbol_base', 'gene_symbol', 'gene_symbols', 'feature_name']:
    if col in adata.var.columns:
        gene_symbols = adata.var[col].astype(str)
        symbol_col = col
        break

if gene_symbols is None:
    gene_symbols = adata.var_names.astype(str)
    symbol_col = 'var_names'

print(f"  Gene symbols from: {symbol_col}")
print(f"  Total genes available: {len(gene_symbols)}")

# Define function to find genes (case-insensitive)
def find_gene_flexible(gene_name, gene_symbols):
    """Find gene with case-insensitive matching"""
    # Try exact match first
    matches = np.where(gene_symbols == gene_name)[0]
    if len(matches) > 0:
        return matches
    
    # Try case-insensitive
    matches = np.where(gene_symbols.str.upper() == gene_name.upper())[0]
    return matches

# Find CD4, CD8A, CD8B
print(f"\nSearching for marker genes:")

cd4_matches = find_gene_flexible(CD4_GENE, gene_symbols)
cd8a_matches = find_gene_flexible(CD8A_GENE, gene_symbols)
cd8b_matches = find_gene_flexible(CD8B_GENE, gene_symbols)

# Check results
if len(cd4_matches) == 0:
    print(f"  ✗ {CD4_GENE}: NOT FOUND")
    raise ValueError(f"Gene {CD4_GENE} not found in adata.var")
else:
    print(f"  ✓ {CD4_GENE}: {len(cd4_matches)} match(es) at index {cd4_matches.tolist()}")

if len(cd8a_matches) == 0:
    print(f"  ✗ {CD8A_GENE}: NOT FOUND")
    raise ValueError(f"Gene {CD8A_GENE} not found in adata.var")
else:
    print(f"  ✓ {CD8A_GENE}: {len(cd8a_matches)} match(es) at index {cd8a_matches.tolist()}")

has_cd8b = len(cd8b_matches) > 0
if has_cd8b:
    print(f"  ✓ {CD8B_GENE}: {len(cd8b_matches)} match(es) at index {cd8b_matches.tolist()}")
else:
    print(f"  ⚠️  {CD8B_GENE}: not found (will use CD8A only)")

# Extract expression from log1p layer (full gene set)
print(f"\nExtracting marker expression from log1p layer...")
print(f"  log1p layer shape: {adata.layers['log1p'].shape}")

# Handle multiple matches by taking max
if len(cd4_matches) > 1:
    print(f"  Multiple {CD4_GENE} matches, using max expression")
    cd4_expr = np.asarray(adata.layers['log1p'][:, cd4_matches]).max(axis=1).flatten()
else:
    cd4_expr = np.asarray(adata.layers['log1p'][:, cd4_matches[0]]).flatten()

if len(cd8a_matches) > 1:
    print(f"  Multiple {CD8A_GENE} matches, using max expression")
    cd8a_expr = np.asarray(adata.layers['log1p'][:, cd8a_matches]).max(axis=1).flatten()
else:
    cd8a_expr = np.asarray(adata.layers['log1p'][:, cd8a_matches[0]]).flatten()

if has_cd8b:
    if len(cd8b_matches) > 1:
        print(f"  Multiple {CD8B_GENE} matches, using max expression")
        cd8b_expr = np.asarray(adata.layers['log1p'][:, cd8b_matches]).max(axis=1).flatten()
    else:
        cd8b_expr = np.asarray(adata.layers['log1p'][:, cd8b_matches[0]]).flatten()
    cd8_expr = np.maximum(cd8a_expr, cd8b_expr)
    print(f"  CD8 score: max(CD8A, CD8B)")
else:
    cd8_expr = cd8a_expr
    print(f"  CD8 score: CD8A only")

# Store in obs
adata.obs['CD4_expr'] = cd4_expr
adata.obs['CD8A_expr'] = cd8a_expr
if has_cd8b:
    adata.obs['CD8B_expr'] = cd8b_expr
adata.obs['CD8_score'] = cd8_expr

print(f"\nExpression statistics:")
print(f"  CD4:  mean={cd4_expr.mean():.2f}, median={np.median(cd4_expr):.2f}, max={cd4_expr.max():.2f}")
print(f"  CD8A: mean={cd8a_expr.mean():.2f}, median={np.median(cd8a_expr):.2f}, max={cd8a_expr.max():.2f}")
if has_cd8b:
    print(f"  CD8B: mean={cd8b_expr.mean():.2f}, median={np.median(cd8b_expr):.2f}, max={cd8b_expr.max():.2f}")
print(f"  CD8 score: mean={cd8_expr.mean():.2f}, median={np.median(cd8_expr):.2f}, max={cd8_expr.max():.2f}")

# Identify T cells
print(f"\nIdentifying T cells (for CD4/CD8 labeling)...")

is_t_cell = adata.obs[CELLTYPIST_KEY].apply(
    lambda x: is_tcell(x, T_CELL_KEYWORDS)
)

n_tcells = is_t_cell.sum()
n_non_tcells = (~is_t_cell).sum()

print(f"  T cells: {n_tcells:,} ({n_tcells/adata.n_obs*100:.1f}%)")
print(f"  Non-T cells (e.g., NK): {n_non_tcells:,} ({n_non_tcells/adata.n_obs*100:.1f}%)")

# Winner-takes-all classification
print(f"\nClassifying T cells (winner-takes-all, threshold: {CD4CD8_THRESHOLD})...")

mask_t = is_t_cell.values
delta = cd4_expr - cd8_expr

# Initialize all as empty string (no prefix)
cd4cd8_type = np.array([''] * adata.n_obs, dtype=object)

# CD4+ : T cell AND CD4 > threshold AND CD4 > CD8
mask_cd4 = mask_t & (cd4_expr > CD4CD8_THRESHOLD) & (delta > 0)

# CD8+ : T cell AND CD8 > threshold AND CD8 > CD4
mask_cd8 = mask_t & (cd8_expr > CD4CD8_THRESHOLD) & (delta < 0)

cd4cd8_type[mask_cd4] = 'CD4+'
cd4cd8_type[mask_cd8] = 'CD8+'

# For non-T cells, keep empty (no CD4/CD8 prefix)
adata.obs[CD4CD8_KEY] = cd4cd8_type
adata.obs['is_t_cell'] = is_t_cell

# Statistics (T cells only)
tcell_cd4cd8 = adata.obs.loc[is_t_cell, CD4CD8_KEY].value_counts()

print(f"\n✓ T cell CD4/CD8 classification (winner-takes-all):")
for cat in ['CD4+', 'CD8+', '']:
    if cat in tcell_cd4cd8.index:
        count = tcell_cd4cd8[cat]
        pct = count / n_tcells * 100
        label = cat if cat else 'Unclear (DP/DN/low)'
        print(f"  {label}: {count:,} cells ({pct:.1f}% of T cells)")

print(f"\n  Non-T cells (no CD4/CD8 label): {n_non_tcells:,}")

# Distribution check
print(f"\nExpression distribution in classified cells:")
if (cd4cd8_type == 'CD4+').sum() > 0:
    cd4_in_cd4pos = cd4_expr[cd4cd8_type == 'CD4+']
    print(f"  CD4+ cells - CD4 expr: mean={cd4_in_cd4pos.mean():.2f}, median={np.median(cd4_in_cd4pos):.2f}")

if (cd4cd8_type == 'CD8+').sum() > 0:
    cd8_in_cd8pos = cd8_expr[cd4cd8_type == 'CD8+']
    print(f"  CD8+ cells - CD8 expr: mean={cd8_in_cd8pos.mean():.2f}, median={np.median(cd8_in_cd8pos):.2f}")

print(f"\n✓ Stage 2 complete")

## STAGE 3: Create Combined Labels (CD4/CD8 + CellTypist)

In [ ]:
print(f"\n{'='*80}")
print("STAGE 3: CREATE COMBINED LABELS")
print("="*80)

print(f"\nCombining CD4/CD8 type with CellTypist labels...")
print(f"  Format: '<CD4/CD8 type> <CellTypist label>'")

# Create combined labels
cd4cd8 = adata.obs[CD4CD8_KEY].astype(str)
celltypist = adata.obs[CELLTYPIST_KEY].astype(str)

combined = cd4cd8 + ' ' + celltypist
adata.obs[COMBINED_LABEL_KEY] = combined

# Show examples
combined_counts = adata.obs[COMBINED_LABEL_KEY].value_counts()
print(f"\n✓ Combined labels created: {len(combined_counts)} unique types")
print(f"\nTop 20 combined cell types:")
for ct, count in combined_counts.head(20).items():
    print(f"  {ct}: {count:,} cells")

# Filter rare types
print(f"\nFiltering rare combined types (min {MIN_CELLS_PER_TYPE} cells)...")
combined_filt = merge_rare_types(
    adata.obs[COMBINED_LABEL_KEY],
    min_cells=MIN_CELLS_PER_TYPE,
    other='Unknown'
)

adata.obs['cell_type_combined_filt'] = combined_filt

# Convert to categorical for scANVI
labels = combined_filt.astype('category')
if 'Unknown' not in labels.cat.categories:
    labels = labels.cat.add_categories(['Unknown'])

adata.obs['labels_for_scanvi'] = labels

# Statistics
n_unknown = int((labels == 'Unknown').sum())
n_labeled = int(adata.n_obs - n_unknown)
n_unique_types = int(labels.nunique()) - (1 if 'Unknown' in labels.cat.categories else 0)

print(f"\n✓ scANVI labels prepared:")
print(f"  Labeled cells: {n_labeled:,} ({n_labeled/adata.n_obs*100:.1f}%)")
print(f"  Unknown cells: {n_unknown:,} ({n_unknown/adata.n_obs*100:.1f}%)")
print(f"  Unique types (excl. Unknown): {n_unique_types}")

# Save counts
print(f"\nSaving combined label counts...")
save_celltype_counts(
    {'combined_labels_filtered': adata.obs['cell_type_combined_filt'].value_counts()},
    OUTPUT_DIR
)

## STAGE 4: Prepare adata_model for scANVI

In [ ]:
print(f"\n{'='*80}")
print("STAGE 4: PREPARE adata_model FOR scANVI")
print("="*80)

print(f"\nCreating adata_model with HVG subset...")
print(f"  (Using genes from original scVI model)")

# Use existing HVG from original training
if 'highly_variable' in adata.var.columns:
    hvg_mask = adata.var['highly_variable'].values
    print(f"  ✓ Found 'highly_variable' column: {hvg_mask.sum()} genes")
else:
    # If no HVG info, use all genes (not recommended for large datasets)
    print(f"  ⚠️  No HVG info found, using all genes")
    hvg_mask = np.ones(adata.n_vars, dtype=bool)

# Build adata_model (minimal construction)
X_hvg = adata.layers['counts'][:, hvg_mask].copy()
obs_hvg = adata.obs[[BATCH_KEY, 'labels_for_scanvi']].copy()
var_hvg = adata.var.loc[hvg_mask, []].copy()

adata_model = sc.AnnData(X=X_hvg, obs=obs_hvg, var=var_hvg)
adata_model.layers['counts'] = adata_model.X.copy()

print(f"\n✓ adata_model created:")
print(f"  Shape: {adata_model.n_obs:,} cells × {adata_model.n_vars:,} genes")
print(f"  obs columns: {list(adata_model.obs.columns)}")
print(f"  .X: raw counts")
print(f"  layers['counts']: raw counts")

# Clean up
del X_hvg, obs_hvg, var_hvg
gc.collect()

## STAGE 5: Load Pre-trained scVI Model

In [ ]:
print(f"\n{'='*80}")
print("STAGE 5: LOAD PRE-TRAINED scVI MODEL")
print("="*80)

# Setup scVI data structure
print(f"\nSetting up scVI on adata_model...")
scvi.model.SCVI.setup_anndata(
    adata_model,
    layer='counts',
    batch_key=BATCH_KEY
)
print(f"✓ scVI data setup complete")

# Load pre-trained model
print(f"\nLoading pre-trained scVI model: {SCVI_MODEL_PATH}")
if not Path(SCVI_MODEL_PATH).exists():
    raise FileNotFoundError(f"scVI model not found: {SCVI_MODEL_PATH}")

vae = scvi.model.SCVI.load(str(SCVI_MODEL_PATH), adata=adata_model)
print(f"✓ scVI model loaded successfully")

# Verify latent representation
print(f"\nVerifying scVI latent representation...")
latent_test = vae.get_latent_representation()
print(f"  Latent shape: {latent_test.shape}")
print(f"  Matches adata X_scvi: {latent_test.shape[0] == adata.n_obs}")

del latent_test

## STAGE 6: Train scANVI with Combined Labels

In [ ]:
print(f"\n{'='*80}")
print("STAGE 6: TRAIN scANVI WITH COMBINED LABELS")
print("="*80)

print(f"\nInitializing scANVI from scVI...")
lvae = scvi.model.SCANVI.from_scvi_model(
    vae,
    adata=adata_model,
    labels_key='labels_for_scanvi',
    unlabeled_category='Unknown'
)
print(f"✓ scANVI initialized")

# Train
print(f"\n{'='*80}")
print(f"Training scANVI...")
print(f"{'='*80}\n")

start_time = time.time()
train_kwargs = {
    'max_epochs': SCANVI_MAX_EPOCHS,
    'batch_size': BATCH_SIZE,
    'train_size': 0.9,
    'accelerator': accelerator,
    'devices': devices,
    'plan_kwargs': {'lr': LEARNING_RATE},
}
if EARLY_STOPPING:
    train_kwargs['early_stopping'] = True
    train_kwargs['early_stopping_patience'] = EARLY_STOPPING_PATIENCE

lvae.train(**train_kwargs)
elapsed = time.time() - start_time
print(f"\n✓ Training complete: {int(elapsed//60)}m {int(elapsed%60)}s")

# Save model
scanvi_model_dir = MODEL_DIR / "scanvi_cd4cd8_model"
print(f"\nSaving scANVI model: {scanvi_model_dir}")
lvae.save(scanvi_model_dir, overwrite=True)

## STAGE 7: Extract scANVI Results

In [ ]:
print(f"\n{'='*80}")
print("STAGE 7: EXTRACT scANVI RESULTS")
print("="*80)

print(f"\nGenerating scANVI predictions (index-aligned)...")

# Predictions
pred = pd.Series(lvae.predict(), index=adata_model.obs_names)
adata.obs[SCANVI_LABEL_KEY] = pred.reindex(adata.obs_names).astype(str).values

# Confidence scores
probs = np.asarray(lvae.predict(soft=True))
conf = pd.Series(probs.max(axis=1), index=adata_model.obs_names)
adata.obs['scanvi_cd4cd8_confidence'] = conf.reindex(adata.obs_names).values

# Latent representation
z = pd.DataFrame(lvae.get_latent_representation(), index=adata_model.obs_names)
adata.obsm['X_scanvi_cd4cd8'] = z.reindex(adata.obs_names).to_numpy()

print(f"✓ scANVI predictions complete")
print(f"  X_scanvi_cd4cd8: {adata.obsm['X_scanvi_cd4cd8'].shape}")
print(f"  Unique types: {adata.obs[SCANVI_LABEL_KEY].nunique()}")
print(f"  Mean confidence: {adata.obs['scanvi_cd4cd8_confidence'].mean():.3f}")

# Filter rare types in predictions
print(f"\nFiltering rare types in scANVI predictions...")
scanvi_filt = merge_rare_types(
    adata.obs[SCANVI_LABEL_KEY],
    min_cells=MIN_CELLS_PER_TYPE,
    other='Unknown'
)
adata.obs['cell_type_scanvi_cd4cd8_filt'] = scanvi_filt.astype('category')

print(f"✓ Unique types (filtered): {adata.obs['cell_type_scanvi_cd4cd8_filt'].nunique()}")

# Save counts
print(f"\nSaving scANVI counts...")
save_celltype_counts(
    {
        'scanvi_cd4cd8_counts_raw': adata.obs[SCANVI_LABEL_KEY].value_counts(),
        'scanvi_cd4cd8_counts_filt': adata.obs['cell_type_scanvi_cd4cd8_filt'].value_counts()
    },
    OUTPUT_DIR
)

# Clean up
del adata_model, vae, lvae
gc.collect()

print(f"\n✓ Stage 7 complete")

## STAGE 8: Visualization

In [ ]:
print(f"\n{'='*80}")
print("STAGE 8: VISUALIZATION")
print("="*80)

# Ensure log1p layer exists
if 'log1p' not in adata.layers:
    print(f"\nCreating log1p layer...")
    from scipy.sparse import issparse
    if issparse(adata.layers['counts']):
        counts_norm = adata.layers['counts'].copy()
        sc.pp.normalize_total(sc.AnnData(X=counts_norm), target_sum=1e4, inplace=True)
        sc.pp.log1p(sc.AnnData(X=counts_norm))
        adata.layers['log1p'] = counts_norm
        del counts_norm
    else:
        adata.layers['log1p'] = adata.layers['counts'].copy()
        sc.pp.normalize_total(sc.AnnData(X=adata.layers['log1p']), target_sum=1e4, inplace=True)
        sc.pp.log1p(sc.AnnData(X=adata.layers['log1p']))
    print(f"✓ log1p layer created")

# Set X for UMAP
adata.X = adata.layers['log1p']

# Compute UMAP on scANVI latent space
print(f"\nComputing UMAP on scANVI latent space...")
sc.pp.neighbors(
    adata,
    use_rep='X_scanvi_cd4cd8',
    n_neighbors=15,
    key_added='neighbors_scanvi_cd4cd8'
)
sc.tl.umap(adata, neighbors_key='neighbors_scanvi_cd4cd8')
adata.obsm['X_umap_scanvi_cd4cd8'] = adata.obsm['X_umap'].copy()
print(f"✓ UMAP computed: {adata.obsm['X_umap_scanvi_cd4cd8'].shape}")

### Plot 1: CD4/CD8 Classification

In [ ]:
print(f"\nGenerating CD4/CD8 classification plot...")
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

sc.pl.embedding(
    adata,
    basis='umap_scanvi_cd4cd8',
    color=CD4CD8_KEY,
    ax=axes[0],
    show=False,
    title='CD4/CD8 Classification',
    size=3,
    legend_loc='right margin',
    frameon=False
)

sc.pl.embedding(
    adata,
    basis='umap_scanvi_cd4cd8',
    color='CD4_expr',
    ax=axes[1],
    show=False,
    title='CD4 Expression',
    size=2,
    cmap='Reds',
    frameon=False
)

sc.pl.embedding(
    adata,
    basis='umap_scanvi_cd4cd8',
    color='CD8A_expr',
    ax=axes[2],
    show=False,
    title='CD8A Expression',
    size=2,
    cmap='Blues',
    frameon=False
)

plt.tight_layout()
plt.savefig(FIG_DIR / 'cd4_cd8_classification.pdf', dpi=300, bbox_inches='tight')
plt.close()
print(f"✓ Saved: cd4_cd8_classification.pdf")

### Plot 2: Combined Labels

In [ ]:
print(f"\nGenerating combined labels plot...")
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sc.pl.embedding(
    adata,
    basis='umap_scanvi_cd4cd8',
    color='cell_type_combined_filt',
    ax=axes[0],
    show=False,
    title='Combined Labels (CD4/CD8 + CellTypist)',
    size=2,
    legend_loc='on data',
    legend_fontsize=6,
    frameon=False
)

sc.pl.embedding(
    adata,
    basis='umap_scanvi_cd4cd8',
    color=BATCH_KEY,
    ax=axes[1],
    show=False,
    title='Batch',
    size=1,
    frameon=False
)

plt.tight_layout()
plt.savefig(FIG_DIR / 'combined_labels.pdf', dpi=300, bbox_inches='tight')
plt.close()
print(f"✓ Saved: combined_labels.pdf")

### Plot 3: scANVI Final Results

In [ ]:
print(f"\nGenerating scANVI final results plot...")
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

sc.pl.embedding(
    adata,
    basis='umap_scanvi_cd4cd8',
    color='cell_type_scanvi_cd4cd8_filt',
    ax=axes[0],
    show=False,
    title='scANVI Final Annotations',
    size=2,
    legend_loc='on data',
    legend_fontsize=6,
    frameon=False
)

sc.pl.embedding(
    adata,
    basis='umap_scanvi_cd4cd8',
    color='scanvi_cd4cd8_confidence',
    ax=axes[1],
    show=False,
    title='scANVI Confidence',
    size=2,
    cmap='viridis',
    frameon=False
)

sc.pl.embedding(
    adata,
    basis='umap_scanvi_cd4cd8',
    color=CD4CD8_KEY,
    ax=axes[2],
    show=False,
    title='CD4/CD8 Type',
    size=2,
    frameon=False
)

plt.tight_layout()
plt.savefig(FIG_DIR / 'scanvi_cd4cd8_final.pdf', dpi=300, bbox_inches='tight')
plt.close()
print(f"✓ Saved: scanvi_cd4cd8_final.pdf")

## STAGE 9: Save Final Results

In [ ]:
print(f"\n{'='*80}")
print("STAGE 9: SAVE FINAL RESULTS")
print("="*80)

output_file = OUTPUT_DIR / "adata_tcell_cd4cd8_FINAL.h5ad"

# Add metadata
adata.uns['pipeline_info'] = {
    'version': '1.0',
    'date': '2025-01-09',
    'input_file': str(INPUT_FILE),
    'batch_key': BATCH_KEY,
    'cd4_cd8_threshold': CD4CD8_THRESHOLD,
    'min_cells_per_type': MIN_CELLS_PER_TYPE,
    'monomac_filtered': True,
    'scvi_model_source': str(SCVI_MODEL_PATH)
}

# Write file
print(f"\nSaving final h5ad file...")
adata.write_h5ad(output_file, compression='gzip')
size_gb = output_file.stat().st_size / 1e9
print(f"✓ Saved: {output_file}")
print(f"   Size: {size_gb:.2f} GB")

# Export annotations
print(f"\nExporting annotations...")
annotations = adata.obs[[
    CELLTYPIST_KEY,
    CD4CD8_KEY,
    'CD4_expr',
    'CD8A_expr',
    COMBINED_LABEL_KEY,
    'cell_type_combined_filt',
    SCANVI_LABEL_KEY,
    'cell_type_scanvi_cd4cd8_filt',
    'scanvi_cd4cd8_confidence',
    BATCH_KEY
]].copy()
annotations.to_csv(OUTPUT_DIR / "annotations_cd4cd8_complete.csv")
print(f"✓ Annotations exported")

## Final Summary

In [ ]:
print(f"\n{'='*80}")
print("🎉 PIPELINE COMPLETE")
print("="*80)

print(f"\n📊 Analysis Summary:")
print(f"   Cells (final): {adata.n_obs:,}")
print(f"   Genes: {adata.n_vars:,}")
print(f"   Batches: {adata.obs[BATCH_KEY].nunique()}")

print(f"\n⭐ CD4/CD8 Classification:")
cd4cd8_summary = adata.obs[CD4CD8_KEY].value_counts()
for cat, count in cd4cd8_summary.items():
    pct = count / adata.n_obs * 100
    print(f"   {cat}: {count:,} cells ({pct:.1f}%)")

print(f"\n⭐ Combined Labels:")
print(f"   Unique types (raw): {adata.obs[COMBINED_LABEL_KEY].nunique()}")
print(f"   Unique types (filtered): {adata.obs['cell_type_combined_filt'].nunique()}")

print(f"\n⭐ scANVI Results:")
print(f"   Unique types (raw): {adata.obs[SCANVI_LABEL_KEY].nunique()}")
print(f"   Unique types (filtered): {adata.obs['cell_type_scanvi_cd4cd8_filt'].nunique()}")
print(f"   Mean confidence: {float(np.mean(adata.obs['scanvi_cd4cd8_confidence'])):.3f}")

print(f"\n📁 Output Files:")
print(f"   h5ad: {output_file.name}")
print(f"   Annotations: annotations_cd4cd8_complete.csv")
print(f"   Cell type counts: *.csv files in {OUTPUT_DIR}")
print(f"   Model: scanvi_cd4cd8_model/")

print(f"\n📈 Figures:")
print(f"   cd4_cd8_classification.pdf - CD4/CD8 type with marker expression")
print(f"   combined_labels.pdf - Combined CD4/CD8 + CellTypist labels")
print(f"   scanvi_cd4cd8_final.pdf - Final scANVI annotations")

print("="*80)
print("\n✨ Next Steps:")
print("   1. Review CD4/CD8 classification results")
print("   2. Validate combined labels with marker genes")
print("   3. Perform differential expression analysis")
print("   4. Explore functional differences between CD4+ and CD8+ subsets")